# Train and evaluate SystemOne Guard

A 756,248-parameter classifier trained from scratch, compared with task-fine-tuned SmolLM2-135M-Instruct with broader task data for Guard and the original task data for frozen SmolLM2. Use the repository development environment; `make setup` installs CPU tools, and the `notebook` extra supplies interactive display packages.

The repository retains actual training traces and complete paired evaluation records. Heavy workflows below are opt-in; saved comparisons can be inspected without downloading model weights.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file()
)
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))
print(ROOT)

## Optional Guard training

Generate the original deterministic synthetic split sizes only if data is absent. The updated guard uses seed 7, dimension 64, hidden size 128, microbatch 32 with accumulation 4, subword features, and three attention layers. `make improve-guard` generates its broader training data and uses validation-only selection. Validation alone controls selection and calibration. Keep the actual generated training trace when replacing published measurements.

In [ ]:
import subprocess

REGENERATE_DATA = False
RETRAIN_GUARD = False

if REGENERATE_DATA:
    subprocess.run(["make", "data"], cwd=ROOT, check=True)

if RETRAIN_GUARD:
    subprocess.run(["make", "setup-gpu"], cwd=ROOT, check=True)
    subprocess.run(["make", "improve-guard"], cwd=ROOT, check=True)

## Inspect both actual training histories

Guard trains four public heads and a private context-intent objective; SmolLM2 LoRA uses response-token loss. The values and historical training runtimes are not directly comparable. Validation quality and NLL are recorded for each run.

In [ ]:
import json

import pandas as pd
from IPython.display import display

for name in ("guard_training", "smollm2_finetuning"):
    trace = json.loads((ROOT / "experiments" / f"{name}.json").read_text())
    print(name)
    display(pd.json_normalize(trace["history"]))

## Run the fine-tuned comparison

Use the companion `Compare_SystemOne_Guard_SmolLM2.ipynb` for detailed quality, calibration, auxiliary, error and latency analysis. The GPU commands require the completed adapter. Set `RETRAIN_SMOLLM` only when recreating it.

In [ ]:
RETRAIN_SMOLLM = False
REEVALUATE_PAIR = False

if RETRAIN_SMOLLM or REEVALUATE_PAIR:
    subprocess.run(["make", "setup-gpu"], cwd=ROOT, check=True)

if RETRAIN_SMOLLM:
    subprocess.run(["make", "finetune"], cwd=ROOT, check=True)

if REEVALUATE_PAIR:
    for split in ("test", "challenge", "manual_probes", "generalization"):
        subprocess.run(
            ["make", "benchmark", f"ARGS=--split {split}"], cwd=ROOT, check=True
        )

## Read verified comparison tables and figures

Absolute accuracy change uses percentage points. Guard latency reduction uses the fine-tuned model's latency as the denominator; speedup uses their ratio. Both models return all four outputs on the same GPU, at batch size 1 for matched request timings. All examples contribute to quality.

In [ ]:
from IPython.display import Image, Markdown
from scripts.plots import create_plots

from systemone_guard.benchmarking.reporting import load_reports, overview

reports = load_reports(ROOT / "experiments")
display(Markdown(overview(reports)))
create_plots(ROOT / "experiments", ROOT / "figures")

for name in (
    "comparison",
    "latency",
    "training",
    "calibration",
    "confusion",
    "auxiliary_accuracy",
):
    display(Image(filename=str(ROOT / "figures" / f"{name}.png")))

## Check changes

Run `make format check`, then `make test-artifacts` when original local data/checkpoints are present. `make plots report` regenerates the published charts and tables from matched records. `make verify-package` checks installation and inference from a built wheel.

High synthetic accuracy is not evidence of real-world security robustness. Keep the manual failures visible and use deterministic authorization independently of this research classifier.